In [0]:
tables = [
    "patients_s",
    "encounters_s",
    "payers_s",
    "procedures_s"
]

for table in tables:
    df = spark.table(f"medical_data.silver.{table}")
    print(f"\nTable: {table}")
    print(df.columns)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_data.gold;

In [0]:
from pyspark.sql import functions as sf

df_patient = spark.table("medical_data.silver.patients_s")

dim_patient = df_patient \
    .withColumnRenamed("id", "patient_id") \
    .select(
        "patient_id",
        "gender",
        "race",
        "ethnicity",
        "marital_status",
        "city_name",
        "state"
    ) \
    .dropDuplicates()

dim_patient.write.mode("overwrite").saveAsTable("medical_data.gold.dim_patient")

In [0]:
df_payer = spark.table("medical_data.silver.payers_s")

dim_payer = df_payer \
    .withColumnRenamed("id", "payer_id") \
    .select(
        "payer_id",
        "name",
        "city",
        "state_headquartered"
    ) \
    .dropDuplicates()

dim_payer.write.mode("overwrite").saveAsTable("medical_data.gold.dim_payer")

In [0]:
df_proc = spark.table("medical_data.silver.procedures_s")

dim_procedure = df_proc \
    .withColumn("procedure_id", sf.monotonically_increasing_id()) \
    .select(
        "procedure_id",
        "encounter_id",
        "code",
        "description"
    ) \
    .dropDuplicates()

dim_procedure.write.mode("overwrite").saveAsTable("medical_data.gold.dim_procedure")

In [0]:
df_enc = spark.table("medical_data.silver.encounters_s")

dim_encounter = df_enc \
    .withColumnRenamed("id", "encounter_id") \
    .select(
        "encounter_id",
        "encounter_class",
        "code",
        "description",
        "organization"
    ) \
    .dropDuplicates()

dim_encounter.write.mode("overwrite").saveAsTable("medical_data.gold.dim_encounter")

In [0]:
df_enc = spark.table("medical_data.silver.encounters_s")
dim_proc = spark.table("medical_data.gold.dim_procedure")

fact_encounters = df_enc \
    .withColumnRenamed("id", "encounter_id") \
    .withColumnRenamed("patient", "patient_id") \
    .withColumnRenamed("payer", "payer_id") \
    .join(dim_proc, "encounter_id", "left") \
    .select(
        "encounter_id",     
        "patient_id",        
        "payer_id",         
        "procedure_id",
        "start",
        "stop",    
        "base_encounter_cost",
        "total_claim_cost",
        "payer_coverage",
        "month",
        "quarter",
        "time_diff"
    )

fact_encounters.write.mode("overwrite").saveAsTable("medical_data.gold.fact_encounters")

In [0]:
%sql 
select * from medical_data.silver.encounters_s
;